In [1]:
from typing import TypedDict

class User(TypedDict):
    # TypedDict의 경우, type hint만을 제공하는 것이라 스키마의 엄밀성은 부족함
    id: int
    name: str
    email: str

In [ ]:
# 데이터 타입에 맞는 값
user1: User = {
    'id': 1,
    'name': 'inheon_choi',
    'email': 'ihchoi@heerae.com'
}

print(user1)

{'id': 1, 'name': 'inheon_choi', 'email': 'ihchoi@heerae.com'}


In [3]:
# 데이터 타입에 맞지 않는 값
user2: User = {
    'id': 1,
    'name': 123, # str 부분에 int 데이터
    'email': 'ihchoi@heerae.com'
}

print(user2)

{'id': 1, 'name': 123, 'email': 'ihchoi@heerae.com'}


In [5]:
from pydantic import BaseModel


class User(BaseModel):
    id: int
    name: str
    email: str

In [ ]:
# 데이터 타입에 맞는 값
user_data_1 = {
    'id': 1,
    'name': 'inheon-choi',
    'email': 'ihchoi@heerae.com'
}

user1 = User(**user_data_1)
print(user1)

id=1 name='inheon-choi' email='ihchoi@heerae.com'


In [7]:
# 데이터 타입에 맞지 않는 값
user_data_2 = {
    'id': 1,
    'name': 123, # 잘못된 값
    'email': 'ihchoi@heerae.com'
}

user2 = User(**user_data_2)
print(user2)

ValidationError: 1 validation error for User
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [8]:
# 그래프 상태에 리듀서 함수 추가하기
# 리듀서: 상태의 업데이트 방식을 지정하는 함수
from typing import TypedDict, Annotated # Annotated로 리듀서 함수 추가함.


def add(left, right):
    return left + right


class State(TypedDict):
    messages: Annotated[list[str], add]

In [ ]:
# langgraph 자체적으로 add_messages 기능이 제공됨.
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages


msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [AIMessage(content="Hi there!", id="2")]

add_messages(msgs1, msgs2)

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='1'),
 AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={}, id='2')]

In [10]:
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [HumanMessage(content="Hello again", id="1")]

add_messages(msgs1, msgs2)

[HumanMessage(content='Hello again', additional_kwargs={}, response_metadata={}, id='1')]

In [11]:
# add_messages는 상태의 메타데이터로 정의되어 리듀서 함수로 사용될 수 있음.
# State 클래스 messages 키의 경우, 새로운 메시지가 추가될 때 기존 메시지 목록에
# 누적되는 방식으로 저장됨을 나타냄.

from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated


class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [12]:
# 그래프 상태에 답변 결과를 반환하기
from typing import TypedDict, Annotated
from operator import add

from langgraph.graph import StateGraph


class State(TypedDict):
    messages: Annotated[list[str], add]
    

graph = StateGraph(State)

In [13]:
def chatbot(state: State):
    question = state["messages"]
    answer = f"사용자 입력을 그대로 반환하는 챗봇입니다. {question}라는 질문을 받았습니다."
    return {"messages": [answer]}


graph.add_node("chatbot", chatbot)

In [14]:
# 시작점과 종료점 엣지 연결하기
from langgraph.graph import START, END

graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

In [ ]:
# 노드 사이에 엣지 추가
graph.add_edge("node_a", "node_b")

In [16]:
# 그래프에 조건부 엣지 추가하기
# chatbot 노드에서 생성한 답변의 길이가 1,000자를 넘어가면 True
# 그렇지 않으면 False를 반환
def routing_function(state: State):
    if len(state["messages"][-1]) > 1000:
        return True
    return False

graph.add_conditional_edges(
    "chatbot",
    routing_function, # 조건 함수 추가
    {True: "summary", False: END}
)